In [8]:
# -*- coding: utf-8 -*-
"""
Scan YAML & Gradle files for Android instrumentation signals and write results.

- Input CSV must contain a 'full_name' column.
- Matches files in All_Config_Files by filename prefix before the first "__".
- For each matched file:
    * YAML (.yml/.yaml): set YML_Check = yes/no + YML_reason
    * Gradle (.gradle/.gradle.kts): set Build_check = yes/no + Build_reason
    * Non-applicable column is set to "NA" with empty reason
- Output Excel is saved next to the input CSV with the name: <input_stem>_check.xlsx
"""

import re
import shutil
from pathlib import Path
from typing import Dict, List, Set

import pandas as pd

# ----------------- CONFIG -----------------
SUPPORT_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\Support")
CONFIG_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files")

# If you know the exact input CSV name, set it here (e.g., "Check.csv").
# Otherwise, the script will auto-pick the most recent CSV in SUPPORT_DIR.
INPUT_CSV_NAME = "Check_Y1_B1_A1.csv"

MAKE_BACKUP_OF_INPUT = False
# ------------------------------------------


# ---- YAML (CI) instrumentation/test signals (raw-text) ----
YAML_SIGNAL_PATTERNS = {
    "emulator_runner_action": r"uses:\s*reactivecircus/android-emulator-runner@",
    "connected_check_task": r"\bconnected(AndroidTest|Check)\b",
    "gradle_connected_task": r"\bgradle[w]?\b[^\n\r]*\bconnected(AndroidTest|Check)\b",
    "managed_device_androidtest_task": r"\b[a-zA-Z0-9_]+(Debug|Release)?AndroidTest\b",
    "adb_instrument": r"\badb\s+shell\s+am\s+instrument\b",
    "gcloud_ftl": r"\bgcloud\b[^\n\r]*\bfirebase\s+test\s+android\s+run\b",
    "detox_android": r"\bnpx\s+detox\b[^\n\r]*(android|--configuration\s+android\.)",
    "avd_create": r"\b(avdmanager|android\s+create\s+avd)\b",
    "emulator_launch": r"\bemulator\s+-avd\b",
    "flutter_android_device_test": r"\bflutter\s+test\b[^\n\r]*\b-d\b[^\n\r]*(emulator|android)",
}

# ---- Gradle/KTS instrumentation build signals ----
BUILD_SIGNAL_PATTERNS = {
    "testInstrumentationRunner": r"\btestInstrumentationRunner\b\s*(=|\s)\s*['\"][^'\"]+['\"]?",
    "androidTestDependency_str": r"\bandroidTest(Implementation|Api|CompileOnly|RuntimeOnly)\s*\(?\s*['\"][^'\"]+['\"]",
    "androidTestDependency_alias": r"\bandroidTest(Implementation|Api|CompileOnly|RuntimeOnly)\s*\(\s*[a-zA-Z0-9_.:-]+\s*\)",
    "androidTestDependency_add": r"\badd\s*\(\s*['\"]androidTest(Implementation|Api|CompileOnly|RuntimeOnly)['\"]\s*,\s*[^)]+\)",
    "androidx_test_dependency": r"['\"][^'\"]*androidx\.test[^'\"]*['\"]",
    "espresso_dependency": r"['\"][^'\"]*espresso[^'\"]*['\"]",
    "uiautomator_dependency": r"['\"][^'\"]*uiautomator[^'\"]*['\"]",
    "orchestrator_dependency": r"['\"][^'\"]*androidx\.test:orchestrator[^'\"]*['\"]",
    "benchmark_dependency": r"['\"][^'\"]*androidx\.benchmark[^'\"]*['\"]",
    "managed_devices": r"\btestOptions\s*\{[^}]*managedDevices\b|testOptions\.managedDevices",
    "useOrchestrator": r"\buseOrchestrator\s*(=|\s)\s*true\b",
    "connectedAndroidTest_task": r"\bconnectedAndroidTest\b",
}

YAML_EXTS = {".yml", ".yaml"}
GRADLE_EXTS = {".gradle", ".gradle.kts"}


def safe_read_text(p: Path) -> str:
    for enc in ("utf-8", "latin-1"):
        try:
            return p.read_text(encoding=enc, errors="ignore")
        except Exception:
            continue
    return ""


def repo_key_from_filename(fname: str) -> str:
    """
    Extract repo key from saved filename '<full_name>__...'.
    """
    base = Path(fname).name
    key = base.split("__", 1)[0] if "__" in base else Path(base).stem
    return key.strip().lower()


def pick_input_csv() -> Path:
    if INPUT_CSV_NAME:
        p = SUPPORT_DIR / INPUT_CSV_NAME
        if p.exists():
            return p
        print(f"[WARN] '{INPUT_CSV_NAME}' not found, falling back to auto-detect.")
    csvs = sorted(SUPPORT_DIR.glob("*.csv"), key=lambda x: x.stat().st_mtime, reverse=True)
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in: {SUPPORT_DIR}")
    for c in csvs:
        if c.name.lower() == "check.csv":
            return c
    return csvs[0]


def index_repo_files(root: Path) -> Dict[str, List[Path]]:
    """
    Index *all* candidate files (.yml/.yaml/.gradle/.gradle.kts) by repo key.
    """
    idx: Dict[str, List[Path]] = {}
    if not root.exists():
        return idx
    for p in root.rglob("*"):
        if p.is_file() and (p.suffix.lower() in YAML_EXTS or p.name.endswith(".gradle.kts") or p.suffix.lower() == ".gradle"):
            key = repo_key_from_filename(p.name)
            idx.setdefault(key, []).append(p)
    return idx


def scan_text(text: str, patterns: Dict[str, str]) -> Set[str]:
    out: Set[str] = set()
    for name, pat in patterns.items():
        if re.search(pat, text, flags=re.IGNORECASE | re.DOTALL):
            out.add(name)
    return out


def main():
    # --- Load input CSV ---
    input_csv = pick_input_csv()
    print(f"[INFO] Using input CSV: {input_csv}")

    if MAKE_BACKUP_OF_INPUT:
        backup = input_csv.with_name(input_csv.stem + "__backup" + input_csv.suffix)
        shutil.copy2(input_csv, backup)
        print(f"[INFO] Backup saved: {backup}")

    df_in = pd.read_csv(input_csv)
    if "full_name" not in df_in.columns:
        raise KeyError("Input CSV must contain a 'full_name' column.")

    # --- Index files by repo key ---
    file_index = index_repo_files(CONFIG_DIR)
    print(f"[INFO] Repos indexed: {len(file_index)}")

    # --- Build output rows ---
    rows: List[dict] = []

    for _, row in df_in.iterrows():
        full_name_raw = str(row["full_name"]).strip()
        repo_key = full_name_raw.lower()

        paths = file_index.get(repo_key, [])
        if not paths:
            continue

        for path in paths:
            text = safe_read_text(path)
            if not text:
                yml_check, yml_reason = ("NA", "")
                build_check, build_reason = ("NA", "")
            else:
                ext = path.suffix.lower()
                is_gradle = (ext == ".gradle") or path.name.endswith(".gradle.kts")
                is_yaml = ext in YAML_EXTS

                # YAML scan
                if is_yaml:
                    yaml_hits = scan_text(text, YAML_SIGNAL_PATTERNS)
                    yml_check = "yes" if yaml_hits else "no"
                    yml_reason = "; ".join(sorted(yaml_hits))
                else:
                    yml_check, yml_reason = ("NA", "")

                # Gradle scan
                if is_gradle:
                    build_hits = scan_text(text, BUILD_SIGNAL_PATTERNS)
                    build_check = "yes" if build_hits else "no"
                    build_reason = "; ".join(sorted(build_hits))
                else:
                    build_check, build_reason = ("NA", "")

            rows.append({
                "full_name": full_name_raw,
                "file_name": path.name,
                "YML_Check": yml_check,
                "YML_reason": yml_reason,
                "Build_check": build_check,
                "Build_reason": build_reason,
            })

    # --- Save output Excel next to the input, named "<input_stem>_check.xlsx" ---
    out_path = input_csv.with_name(input_csv.stem + "_check.xlsx")
    out_df = pd.DataFrame(rows, columns=[
        "full_name", "file_name", "YML_Check", "YML_reason", "Build_check", "Build_reason"
    ])
    out_df.to_excel(out_path, index=False)
    print(f"[DONE] Wrote: {out_path}  (rows: {len(out_df)})")


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")


[INFO] Using input CSV: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\Support\Check_Y1_B1_A1.csv
[INFO] Repos indexed: 4519
[DONE] Wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\Support\Check_Y1_B1_A1_check.xlsx  (rows: 999)
